In [ ]:
import pandas as pd
import numpy as np
import optuna
import joblib
import xgboost as xgb
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import PowerTransformer

In [ ]:
def preprocess_data(df):
    # Target Encoding
    target_encoders = {}

    for col in ['station_id', 'station', 'state', 'city']:
        if col in df.columns:
            target_encoders[col] = (
                df.groupby(col)['sr_wm2']
                .mean()
                .to_dict()
            )

    global_mean = df['sr_wm2'].mean()

    for col in ['station_id', 'station', 'state', 'city']:
        if col in df.columns:
            df[f'{col}_encoded'] = df[col].map(target_encoders[col])
            df[f'{col}_encoded'] = (
                df[f'{col}_encoded']
                .fillna(global_mean)
            )

    # Date Features
    df['timestamp'] = pd.to_datetime(df['timestamp'])

    df = df.sort_values("timestamp").reset_index(drop=True)

    df["year"] = df["timestamp"].dt.year
    df["month"] = df["timestamp"].dt.month
    df["day"] = df["timestamp"].dt.day
    df["hour"] = df["timestamp"].dt.hour

    # 3. Drop Redundant Columns
    cols_to_drop = [
        'timestamp',
        'at_c',
        'rh_pct',
        'ws_ms',
        'wd_deg',
        'rf_mm',
        'tot_rf_mm',
        'station_id',
        'station',
        'state',
        'city',
        'era5_temp_k',
        'era5_dewpoint_k',
        'era5_pressure_pa',
        'era5_u10_ms',
        'era5_v10_ms',
        'era5_precip_m',
        'solar_altitude_deg',
        'era5_sw_down_wm2'
    ]

    df.drop(
        columns=[c for c in cols_to_drop if c in df.columns],
        inplace=True
    )

    # Cyclical Encoding
    df['hour_sin'] = np.sin(
        df['hour'] * (2. * np.pi / 24)
    ).astype('float32')

    df['hour_cos'] = np.cos(
        df['hour'] * (2. * np.pi / 24)
    ).astype('float32')

    df['month_sin'] = np.sin(
        (df['month'] - 1) * (2. * np.pi / 12)
    ).astype('float32')

    df['month_cos'] = np.cos(
        (df['month'] - 1) * (2. * np.pi / 12)
    ).astype('float32')

    df.drop(columns=['hour', 'month'], inplace=True)

    # Lag Features
    columns_to_ignore = [
        'station_id_encoded',
        'latitude',
        'longitude',
        'solar_zenith_deg',
        'cos_zenith',
        'hour_sin',
        'hour_cos',
        'month_sin',
        'month_cos',
        'year'
    ]

    columns_to_lag = [
        col for col in df.columns
        if col not in columns_to_ignore
    ]

    df = df.sort_values(
        by=['station_id_encoded', 'year', 'hour_sin']
    )

    for col in columns_to_lag:

        df[f'{col}_lag1'] = (
            df.groupby('station_id_encoded')[col]
            .shift(1)
            .astype('float32')
        )

        df[f'{col}_lag24'] = (
            df.groupby('station_id_encoded')[col]
            .shift(24)
            .astype('float32')
        )

    return df.dropna().astype({
        c: 'float32'
        for c in df.select_dtypes('float64').columns
    })

In [ ]:
def train_pipeline(df):
    pt = PowerTransformer(method='yeo-johnson')

    df['sr_wm2_scaled'] = pt.fit_transform(
        df[['sr_wm2']]
    ).astype('float32')

    train_df = df[df['year'] < 2025]
    test_df = df[df['year'] >= 2025]

    features = [
        col for col in df.columns
        if col not in ['sr_wm2', 'sr_wm2_scaled', 'year']
    ]

    X_train = train_df[features]
    y_train = train_df['sr_wm2_scaled']

    X_test = test_df[features]
    y_test = test_df['sr_wm2_scaled']

    dtrain = xgb.DMatrix(X_train, label=y_train)
    dtest = xgb.DMatrix(X_test, label=y_test)


    def objective(trial):

        params = {
            "objective": "reg:squarederror",
            "eval_metric": "rmse",
            "device": "cuda",

            "learning_rate": trial.suggest_float(
                "learning_rate",
                0.01,
                0.2,
                log=True
            ),

            "max_depth": 6,
            "subsample": 0.8
        }

        model = xgb.train(
            params,
            dtrain,
            num_boost_round=300,
            evals=[(dtest, "val")],
            early_stopping_rounds=20,
            verbose_eval=False
        )

        return model.best_score

    print("Optimizing Learning Rate...")

    study = optuna.create_study(direction="minimize")

    study.optimize(objective, n_trials=10)


    print(
        f"Best learning rate: "
        f"{study.best_params['learning_rate']}"
    )

    best_params = {
        "objective": "reg:squarederror",
        "eval_metric": "rmse",
        "device": "cuda",

        "learning_rate": study.best_params["learning_rate"],

        "max_depth": 8,
        "subsample": 0.9
    }

    final_model = xgb.train(
        best_params,
        dtrain,
        num_boost_round=1000,
        evals=[(dtest, "val")],
        early_stopping_rounds=50,
        verbose_eval=100
    )

    y_pred_scaled = final_model.predict(dtest)

    # Inverse transform back to original scale
    y_pred = pt.inverse_transform(
        y_pred_scaled.reshape(-1, 1)
    ).flatten()

    y_true = pt.inverse_transform(
        y_test.values.reshape(-1, 1)
    ).flatten()

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))

    mae = mean_absolute_error(y_true, y_pred)

    r2 = r2_score(y_true, y_pred)

    print("\n========== FINAL METRICS ==========")
    print(f"RMSE : {rmse:.4f}")
    print(f"MAE  : {mae:.4f}")
    print(f"R2   : {r2:.4f}")


    joblib.dump(final_model, "xgb_optimized_model.pkl")
    joblib.dump(pt, "power_transformer.pkl")

    print("\nPipeline complete.")
    print("Model + transformer saved.")

In [ ]:
data_path = '/kaggle/input/datasets/prajyotr/mlpr-project-dataset/dataset.csv'
df = pd.read_csv(data_path)
df_clean = preprocess_data(df)
train_pipeline(df_clean)
print("Training complete.")